<a href="https://colab.research.google.com/github/areebazia-lsh/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## Method choice

I will use a **Random Forest classifier** for this lane.

The goal is to rank pages that are more likely to be declining so that content teams can decide which pages to review first. A Random Forest can combine several observable signals at the same time, such as impressions, CTR, average position, content age, and engagement.

This is a useful next step after the Week-4 hand rule because the baseline uses fixed thresholds, while the model can learn non-linear combinations of the available signals.

I will evaluate the model using Precision@50 because the practical decision is which pages should appear at the top of the review queue.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

## Split design

I will use a **client-grouped train/test split**.

Pages from the same client should not appear in both training and test data. Otherwise, the model could benefit from client-specific patterns that would make the evaluation look better than it really is.

I will keep approximately 20% of the clients for the test set and use the remaining clients for training. The Week-4 baseline will be evaluated on the exact same test rows and using the same Precision@50 metric.

In [19]:
import os
print(os.getcwd())
print(os.listdir("/content"))

/content/flyrank-ml-internship
['.config', 'flyrank-ml-internship', 'sample_data']


In [20]:
!git clone https://github.com/areebazia-lsh/flyrank-ml-internship.git

fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.


In [21]:
%cd /content/flyrank-ml-internship

/content/flyrank-ml-internship


In [22]:
!ls data/raw

content_refresh_anonymized.csv


In [23]:
import pandas as pd
import numpy as np
import os

from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import precision_score

# Load starter data
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Rows:", len(df))
print("Clients:", df["client_id"].nunique())

# Observed target
y = (
    df["trend_direction"]
    .str.lower()
    .eq("down")
    .astype(int)
)

# Features known before the decision
features = [
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "engagement_rate",
    "content_age_days",
    "word_count"
]

X = df[features].copy()

# Grouped split: clients never cross train/test
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=df["client_id"])
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Train rows:", len(X_train))
print("Test rows:", len(X_test))
print("Train clients:", df.iloc[train_idx]["client_id"].nunique())
print("Test clients:", df.iloc[test_idx]["client_id"].nunique())

print(
    "Client overlap:",
    len(
        set(df.iloc[train_idx]["client_id"])
        & set(df.iloc[test_idx]["client_id"])
    )
)

Rows: 30000
Clients: 32
Train rows: 23837
Test rows: 6163
Train clients: 25
Test clients: 7
Client overlap: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [24]:
# Model pipeline
model = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "random_forest",
        RandomForestClassifier(
            n_estimators=300,
            max_depth=8,
            min_samples_leaf=5,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        )
    )
])

# Train only on the training clients
model.fit(X_train, y_train)

# Model score on untouched test clients
model_scores = model.predict_proba(X_test)[:, 1]

In [25]:
def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))[:k]
    return np.asarray(labels)[order].mean()

# ----- MODEL -----
model_p50 = precision_at_k(
    model_scores,
    y_test,
    50
)

# ----- WEEK-4 BASELINE -----
test_df = df.iloc[test_idx].copy()

test_df["is_stale"] = (
    test_df["days_since_last_update"] >= 180
).astype(int)

test_df["is_visible"] = (
    test_df["impressions_90d"] >= 500
).astype(int)

test_df["is_ctr_opportunity"] = (
    test_df["avg_position"].between(
        1, 20, inclusive="both"
    )
    & (test_df["ctr"] < 0.30)
).astype(int)

test_df["baseline_score"] = (
    2 * test_df["is_stale"]
    + 2 * test_df["is_ctr_opportunity"]
    + test_df["is_visible"]
)

baseline_p50 = precision_at_k(
    test_df["baseline_score"],
    y_test,
    50
)

comparison = pd.DataFrame({
    "Method": [
        "Week-4 baseline",
        "Random Forest"
    ],
    "Precision@50": [
        baseline_p50,
        model_p50
    ]
})

comparison

,Method,Precision@50
0,Week-4 baseline,0.72
1,Random Forest,0.56


## Model vs Baseline Interpretation

The Random Forest achieved a Precision@50 of **0.56** on the held-out clients, while the Week-4 baseline achieved **0.72**.

Both methods were evaluated on the same held-out test rows using the same Precision@50 metric. In this run, the Random Forest performed worse than the Week-4 baseline.

This is a measured result on the available test data. It suggests that the simple Week-4 rule was more effective for the top-50 ranking in this split. The result does not prove that the baseline will always outperform the model on future data.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [26]:
# Test-set predictions
test_results = test_df.copy()

test_results["model_score"] = model_scores
test_results["actual_declining"] = y_test.values

test_results["predicted_top50"] = 0

top50_idx = (
    test_results["model_score"]
    .sort_values(ascending=False)
    .head(50)
    .index
)

test_results.loc[top50_idx, "predicted_top50"] = 1

# False positives among top 50
false_positives = test_results[
    (test_results["predicted_top50"] == 1)
    & (test_results["actual_declining"] == 0)
]

print("False positives in top 50:", len(false_positives))

print(
    false_positives[
        [
            "content_id",
            "impressions_90d",
            "ctr",
            "avg_position",
            "content_age_days"
        ]
    ].head(10)
)

False positives in top 50: 22
                 content_id  impressions_90d   ctr  avg_position  \
1394   content_cc088a55deca              211  0.00          27.9   
2150   content_3cd58a4a2731             2004  0.00          39.1   
2357   content_8f1409b2674e              209  0.00          20.0   
4050   content_500bd3907331             4037  0.10           5.5   
5011   content_c148e44db30d              335  0.00          31.3   
5477   content_3164f3076003             2696  0.04          16.1   
7599   content_0e52d032d7f7             1172  0.17           6.5   
10080  content_35d63627bf3e             1525  0.00          32.6   
11061  content_0b47dae0c7f9             1191  0.00          23.1   
12069  content_ff4370afd49c             1677  0.18          33.1   

       content_age_days  
1394                275  
2150                272  
2357                271  
4050                230  
5011                275  
5477                275  
7599                228  
10080        

In [27]:
rf = model.named_steps["random_forest"]

importance = pd.DataFrame({
    "feature": features,
    "importance": rf.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

print(importance.to_string(index=False))

               feature  importance
       impressions_90d    0.291772
      content_age_days    0.225350
          avg_position    0.218345
            word_count    0.111030
                   ctr    0.068936
days_since_last_update    0.064046
       engagement_rate    0.020521


In [28]:
print("Model Precision@50:", model_p50)
print("Baseline Precision@50:", baseline_p50)

print("\nTop-50 model errors:")
print("False positives:", len(false_positives))
print("True positives:", 50 - len(false_positives))

Model Precision@50: 0.56
Baseline Precision@50: 0.72

Top-50 model errors:
False positives: 22
True positives: 28


## Interpretation

The top-ranked pages should be treated as decision-support recommendations rather than guaranteed refresh candidates.

The error analysis shows that some pages in the top 50 can still be non-declining. For example, a page may have a low CTR or old content but have a valid reason for that pattern that is not represented in the available features.

Feature importance helps show which signals the Random Forest relied on most in this run. These are measured model signals, not proof that a feature causes performance changes.

The main question is whether the model improves Precision@50 over the Week-4 baseline on unseen clients, not whether the model is simply more complex.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.